# B2 · The price depends on where you are: KNN regressor on latitude and longitude

**Type**: Implement with scikit-learn · **Level**: ●●○ independent · **Time**: about 55 minutes · **core path ★**

## Objective

Compare a linear regression with several KNN regressors on house prices in California, show that the KNN with **only latitude and longitude** beats the linear model with all the features, and
that a distance with **per-feature weights** (small weight on income) beats both even more. The weight is chosen on a validation set, not on the test set.

## Dataset

`ds.california()` (20,640 districts, 8 features: `MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude`; target: median price in hundreds of thousands of dollars): 75/25 split with `random_state=0`.
Error: RMSE (`mean_squared_error(..., squared=False)`).

## Educational purpose

Understand that KNN is a model with a **proximity structure** that must be chosen: for house prices geographic proximity is almost everything. And see the effect of choosing the weights of the features in the distance, which is
the simplest way to "teach" KNN which columns matter.

## Task

1. Compute the test RMSE of five models and save it in `rmse` (dictionary): `linear` (`LinearRegression`, 8 features), `raw knn` (`KNeighborsRegressor(15)`, 8 features **without** scaler), `scaled knn` (with `StandardScaler`, 8 features), `geo knn` (`KNeighborsRegressor(15)` on `Latitude` and `Longitude` only), `geo + income knn` (the same two plus `MedInc`, scaled).
2. `GridSearchCV` (5 folds, `scoring="neg_root_mean_squared_error"`) on `Latitude` and `Longitude` with `n_neighbors` in `[3, 5, 10, 15, 25, 40, 60]` and `weights` in `["uniform", "distance"]`: save `geo_params` and `best_geo_rmse` (test RMSE).
3. **Per-feature weights.** Standardize `Latitude`, `Longitude`, `MedInc` (scaler learned on the training set) and multiply the income column by a weight `w`. Choose `w` among `[0, 0.05, 0.1, 0.2, 0.35, 0.5, 1]` with a validation set (20% of the training set, `train_test_split(random_state=1)`), with `KNeighborsRegressor(10, weights="distance")`: save `rmse_per_w` (`pd.Series` on the validation set) and `best_w`.
4. With that `w`, retrain on the **whole** training set and save `weighted_rmse` (test RMSE).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = ds.california()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)
GEO = ["Latitude", "Longitude"]


def rmse_of(model, columns):
    return mean_squared_error(y_te, model.fit(X_tr[columns], y_tr).predict(X_te[columns]), squared=False)

## Your work

In [ ]:
rmse = {}
rmse

In [ ]:
geo_params = None
best_geo_rmse = None

In [ ]:
cols = GEO + ["MedInc"]
rmse_per_w = pd.Series(dtype=float)
best_w = None
weighted_rmse = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='Weights: `sc = StandardScaler().fit(X_fit[cols]); A = sc.transform(X_fit[cols]); A[:, 2] *= w`. Apply the same `w` to the validation set. After the choice: retrain on the whole `X_tr` with a scaler learned on all of `X_tr`.'):
    check(set(rmse) == {"linear", "raw knn", "scaled knn", "geo knn", "geo + income knn"}, "five models compared", "keys: linear, raw knn, scaled knn, geo knn, geo + income knn")
    check(rmse["raw knn"] > rmse["linear"] and rmse["scaled knn"] < rmse["linear"], "without scaling the KNN is worse than the linear one; with scaling better", "Population (scale 1000) dominates the raw distance")
    check(rmse["geo knn"] < rmse["scaled knn"] - 0.05 and rmse["geo knn"] < rmse["linear"] - 0.15, "the KNN with just two coordinates beats the one with 8 features and the linear one (by over 0.15)", "fewer features but the right ones")
    check(rmse["geo + income knn"] >= rmse["geo knn"] - 0.02, "adding the income with the same weight as the coordinates does not improve (or barely)", "the standardized income weighs as much as latitude and longitude")
    check(geo_params["n_neighbors"] <= 15 and best_geo_rmse < 0.55, "the search on the coordinates alone chooses a small k (15 or less); RMSE below 0.55", "GridSearchCV on Latitude and Longitude")
    check(list(rmse_per_w.index) == [0, 0.05, 0.1, 0.2, 0.35, 0.5, 1] and best_w == rmse_per_w.idxmin() and 0 < best_w <= 0.2, "the best weight of the income is small but nonzero (between 0.05 and 0.2)", "try the seven weights on the validation set")
    check(weighted_rmse < rmse["geo knn"] - 0.03 and weighted_rmse < best_geo_rmse, "with the weights, the test RMSE drops by over 0.03 compared to the coordinates alone and also beats the search on k", "retrain with best_w on the whole training set")
done()

---
The commented solution is in `solutions/B_with_sklearn/B2_knn_regressor_and_geography.ipynb`. Open it after you pass the check, or after you have really tried.